In [3]:
import pandas as pd 
import numpy as np
from faker import Faker
import random

fake = Faker()

#Number of patients

num_patients = 50000

insurance_list = [
    "Aetna",
    "Blue Cross",
    "Cigna",
    "United Healthcare",
    "Medicure"
]

chronic_conditions = [
    "Diabetes",
    "Hypertension",
    "Heart Disease",
    "Asthma",
    "None"
]

patients = []

for i in range(1, num_patients + 1):

    patient = {
        "patient_id": i,
        "patient_name": fake.name(),
        "age": random.randint(18,90),
        "gender": random.choice(["Male", "Female"]),
        "city": fake.city(),
        "insurance_provider": random.choice(insurance_list),
        "chronic_condition": random.choice(chronic_conditions),
        "registration_date": fake.date_between(
            start_date= '-3y',
            end_date = 'today'
        )
    }

    patients.append(patient)

#convert to dataframe

patients_df = pd.DataFrame(patients)

#save csv

patients_df.to_csv(r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\patients.csv", index = False)

print("patients.csv created sucessfully")
print(patients_df.head())
    

patients.csv created sucessfully
   patient_id      patient_name  age  gender                city  \
0           1       Mary Miller   41  Female  Port Alexanderland   
1           2       Tracy Jones   42    Male   Port Brianchester   
2           3    Shannon Palmer   84    Male           Jerrystad   
3           4  Melissa Williams   40  Female         Tiffanyland   
4           5  Krystal Martinez   45    Male   Port Jesusborough   

  insurance_provider chronic_condition registration_date  
0              Cigna          Diabetes        2026-03-11  
1         Blue Cross     Heart Disease        2025-06-10  
2         Blue Cross            Asthma        2024-07-18  
3              Aetna            Asthma        2025-08-01  
4              Cigna     Heart Disease        2023-12-10  


In [10]:
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker()

# Load patients
patients_df = pd.read_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\patients.csv"
)

# Convert registration_date to datetime
patients_df['registration_date'] = pd.to_datetime(patients_df['registration_date'])

# Number of visits
num_visits = 200000

# Step 1: Generate base visits with random patient IDs
visits_df = pd.DataFrame({
    "visit_id": range(1, num_visits + 1),
    "patient_id": np.random.choice(patients_df['patient_id'], num_visits),
    "doctor_id": np.random.randint(1001, 1101, num_visits),
    "treatment_cost": np.round(np.random.uniform(100, 15000, num_visits), 2),
    "payment_status": np.random.choice(["Paid", "Pending", "Denied"], num_visits)
})

# Step 2: Merge patient info (registration_date + chronic_condition)
visits_df = visits_df.merge(
    patients_df[['patient_id', 'registration_date', 'chronic_condition']],
    on='patient_id',
    how='left'
)

# Step 3: Ensure visit_date >= registration_date
visits_df['visit_date'] = visits_df.apply(
    lambda row: fake.date_between(
        start_date=row['registration_date'].date(),
        end_date='today'
    ),
    axis=1
)

# Step 4: Define mapping
condition_map = {
    "Diabetes": {"department": "General Medicine", "diagnosis": "Diabetes"},
    "Hypertension": {"department": "General Medicine", "diagnosis": "Hypertension"},
    "Heart Disease": {"department": "Cardiology", "diagnosis": "Chest Pain"},
    "Asthma": {"department": "Pulmonology", "diagnosis": "Asthma"},
    "None": {"department": "General Medicine", "diagnosis": "Fever"}
}

# Step 5: Apply mapping
visits_df['department'] = visits_df['chronic_condition'].map(
    lambda cond: condition_map.get(cond, {}).get('department', "General Medicine")
)
visits_df['diagnosis'] = visits_df['chronic_condition'].map(
    lambda cond: condition_map.get(cond, {}).get('diagnosis', "Fever")
)

# Step 6: Randomness override (5% chance)
randomness_probability = 0.05  # 5% random, 95% correct
mask = np.random.rand(len(visits_df)) < randomness_probability

visits_df.loc[mask, 'department'] = np.random.choice(
    ["Cardiology", "Orthopedics", "Neurology", "Emergency", "Pediatrics", "Oncology", "General Medicine"],
    mask.sum()
)
visits_df.loc[mask, 'diagnosis'] = np.random.choice(
    ["Diabetes", "Hypertension", "Fracture", "Chest Pain", "Migraine", "Asthma", "Fever", "Cancer"],
    mask.sum()
)

# Step 7: Drop helper columns (registration_date + chronic_condition)
visits_df = visits_df.drop(columns=['registration_date', 'chronic_condition'])

# Step 8: Save CSV
visits_df.to_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\visits.csv",
    index=False
)

print("visits.csv created successfully")
print(visits_df.head())


visits.csv created successfully
   visit_id  patient_id  doctor_id  treatment_cost payment_status  visit_date  \
0         1       27066       1049        12401.69         Denied  2026-03-25   
1         2        5905       1028         9702.81           Paid  2025-01-14   
2         3       35321       1013        10056.43        Pending  2026-03-03   
3         4       23880       1059        11618.54           Paid  2025-06-16   
4         5       14367       1020         8300.95         Denied  2026-02-28   

         department     diagnosis  
0  General Medicine      Diabetes  
1  General Medicine  Hypertension  
2  General Medicine         Fever  
3        Cardiology    Chest Pain  
4  General Medicine         Fever  


In [7]:
import pandas as pd
import numpy as np
import random

# Load visits
visits_df = pd.read_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\visits.csv"
)

num_claims = 120000

claim_status_list = ["Approved", "Pending", "Denied"]
denial_reasons = ["Incomplete Documents", "Policy Expired", "Coverage Issue",
                  "Duplicate Claim", "Preauthorization Missing", "None"]

# Sample claims from visits
sampled_visits = visits_df.sample(num_claims)[['visit_id','patient_id']].reset_index(drop=True)

claims = []
for i in range(num_claims):
    visit_id = sampled_visits.loc[i, 'visit_id']
    patient_id = sampled_visits.loc[i, 'patient_id']
    claim_amount = round(random.uniform(500, 20000), 2)
    claim_status = random.choice(claim_status_list)

    if claim_status == "Approved":
        approved_amount = round(claim_amount * random.uniform(0.7, 1.0), 2)
        denial_reason = "None"
    elif claim_status == "Pending":
        approved_amount = 0
        denial_reason = "None"
    else:
        approved_amount = 0
        denial_reason = random.choice(denial_reasons[:-1])  # exclude "None"

    claim = {
        "claim_id": i+1,
        "visit_id": visit_id,
        "patient_id": patient_id,
        "claim_amount": claim_amount,
        "approved_amount": approved_amount,
        "claim_status": claim_status,
        "denial_reason": denial_reason,
        "days_to_settlement": random.randint(1, 90)
    }
    claims.append(claim)

claims_df = pd.DataFrame(claims)
claims_df.to_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\claims.csv",
    index=False
)
print("claims.csv created successfully")
print(claims_df.head())


claims.csv created successfully
   claim_id  visit_id  patient_id  claim_amount  approved_amount claim_status  \
0         1     80872       45923       6996.19              0.0       Denied   
1         2    195043       39499       1308.40              0.0       Denied   
2         3    166090       29585       7777.64              0.0       Denied   
3         4    142861       17178      18283.17              0.0      Pending   
4         5    146573       35539      13480.37              0.0       Denied   

              denial_reason  days_to_settlement  
0      Incomplete Documents                  88  
1           Duplicate Claim                  65  
2  Preauthorization Missing                  30  
3                      None                  36  
4            Policy Expired                   2  


In [9]:
import pandas as pd
import random
from faker import Faker

fake = Faker()

# Load visits
visits_df = pd.read_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\visits.csv"
)

# Convert visit_date to datetime
visits_df['visit_date'] = pd.to_datetime(visits_df['visit_date'])

num_followups = 100000

# Sample followups from visits
sampled_visits = visits_df.sample(num_followups)[['visit_id','patient_id','visit_date']].reset_index(drop=True)

followups = []
for i in range(num_followups):
    visit_id = sampled_visits.loc[i, 'visit_id']
    patient_id = sampled_visits.loc[i, 'patient_id']
    visit_date = sampled_visits.loc[i, 'visit_date']

    missed_followup = random.choice(["Yes", "No"])

    if missed_followup == "Yes":
        readmission_flag = random.choices(["Yes", "No"], weights=[70, 30])[0]
    else:
        readmission_flag = random.choices(["Yes", "No"], weights=[20, 80])[0]

    record = {
        "followup_id": i+1,
        "visit_id": visit_id,
        "patient_id": patient_id,
        # ✅ convert visit_date to datetime.date before passing to Faker
        "followup_date": fake.date_between(start_date=visit_date.date(), end_date='today'),
        "missed_followup": missed_followup,
        "readmission_flag": readmission_flag
    }
    followups.append(record)

followups_df = pd.DataFrame(followups)
followups_df.to_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\raw\followups.csv",
    index=False
)
print("followups.csv created successfully")
print(followups_df.head())


followups.csv created successfully
   followup_id  visit_id  patient_id followup_date missed_followup  \
0            1     56715       47064    2026-02-17              No   
1            2        84       13403    2025-04-23              No   
2            3    150460       41498    2026-05-12             Yes   
3            4      1183       38604    2026-04-26             Yes   
4            5    178612       38459    2025-12-30              No   

  readmission_flag  
0               No  
1               No  
2               No  
3              Yes  
4               No  
